In [1]:
import pandas as pd
import os
from models.gender_classifier import GenderClassifier
from models.gender_rewrite import GenderRewrite
from tqdm import tqdm
import json
from misc import NameAnonymizer


In [2]:
pd.set_option('display.max_colwidth', None)


In [3]:
data_dir = "./data"
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")

raw_dir = os.path.join(data_dir, "structured", "situations")
processed_dir = os.path.join(data_dir, "processed")


In [4]:
name_anonymizer = NameAnonymizer(
    names_path=spanish_names_path,
    whitelist_path=name_whitelist_path,
    replacement="{{name}}"
)


In [5]:
name_anonymizer.anonymize_names("Hola, me llamo Juan y soy Juan")


'Hola, me llamo {{name}} y soy {{name}}'

In [6]:
class SituationProcessor:
    NEUTRAL_LABEL = "neutral"

    gender_map = {
        "male": "masculino",
        "female": "femenino",
    }

    opposite_gender_map = {
        "masculino": "femenino",
        "femenino": "masculino",
    }

    def __init__(self, raw_dir: str, processed_dir: str, metadata_path: str, name_anonymizer: NameAnonymizer, threshold: float = 0.98):
        self.raw_dir = raw_dir
        self.processed_dir = processed_dir
        self.name_anonymizer = name_anonymizer
        self.threshold = threshold

        os.makedirs(self.processed_dir, exist_ok=True)

        with open(metadata_path, "r", encoding="utf-8") as f:
            self.metadata = json.load(f)
			
        self.clf = GenderClassifier()
        self.chain = GenderRewrite()

    def process_situation(self, number: int) -> pd.DataFrame:
        filename = f"situation_{number}.csv"

        raw_path = os.path.join(
            self.raw_dir, 
            filename
        )

        processed_path = os.path.join(
            self.processed_dir,
            filename,
        )

        print(f"\nProcesando situación {number}")
        print(f"Archivo: {raw_path}")

        df = pd.read_csv(raw_path)

        metadata = self.metadata.get(str(number), {})

        choices_map = metadata.get("choices", {})

        if "choice" in df.columns:
            df["choice"] = (
                df["choice"]
                .map(choices_map)
                .fillna(df["choice"])
            )

        # Eliminar respuestas inválidas
        remove_texts = metadata.get("remove", [])
        mask_remove = df["text"].isin(remove_texts)

        removed_rows = df[mask_remove]

        print(f"Filas eliminadas por metadata: {len(removed_rows)}")

        df = df[~mask_remove]

        # Eliminar filas con campos libres nulos
        df = df[df["text"].notna()].copy()

        # Sustituir nombres de personas por "{{name}}"
        df["text_clean"] = df["text"].apply(self.name_anonymizer.anonymize_names)

        before = len(df)

        subset_cols = ["text_clean"]

        if "choice" in df.columns:
            subset_cols.append("choice")

        # Deduplicar columnas
        df = df.drop_duplicates(
            subset=subset_cols,
            keep="first"
        )

        after = len(df)

        print(f"Duplicados eliminados: {before - after}")

        text_clean = df["text_clean"].tolist()

        clf_results = self.clf.predict_batch(
            text_clean,
            return_all_scores=True,
        )

        flipped_texts = []
        flip_flags = []
        confidences = []

        iterator = zip(text_clean, clf_results)

        for text, res in tqdm(iterator, total=len(text_clean)):
            best_label = res["label"]
            best_conf = res["confidence"]

            should_flip = (
                best_label != self.NEUTRAL_LABEL
                and best_conf >= self.threshold
            )

            if should_flip:
                detected_gender = self.gender_map[best_label]
                target_gender = self.opposite_gender_map[detected_gender]

                flipped = self.chain.rewrite(
                    text,
                    source_gender=detected_gender,
                    target_gender=target_gender,
                )

            else:
                flipped = text

            flipped_texts.append(flipped)
            flip_flags.append(should_flip)
            confidences.append(best_conf)

        df["text_flipped"] = flipped_texts
        df["was_flipped"] = flip_flags
        df["gender_confidence"] = confidences

        # Se vuelve a anonimizar por si el LLM ha producido algún cambio raro
        df["text_flipped"] = (
            df["text_flipped"]
            .apply(self.name_anonymizer.anonymize_names)
        )

        df.to_csv(processed_path, index=False)

        print(f"Guardado: {processed_path}")

        return df

    def process_situations(self, numbers: list[int]) -> dict[int, pd.DataFrame]:
        results = {}

        for number in numbers:
            try:
                clean_df = self.process_situation(number)
                results[number] = clean_df

            except Exception as e:
                print(f"Error procesando situación {number}: {e}")

        return results


In [7]:
processor = SituationProcessor(
    raw_dir=raw_dir,
    processed_dir=processed_dir,
    metadata_path="./metadata.json",
    name_anonymizer=name_anonymizer
)


In [10]:
df = processor.process_situation(1)




Procesando situación 1
Archivo: ./data\structured\situations\situation_1.csv
Filas eliminadas por metadata: 2
Duplicados eliminados: 1


100%|██████████| 38/38 [00:32<00:00,  1.16it/s]

Guardado: ./data\processed\situation_1.csv


In [11]:
display(df[["text", "text_clean", "text_flipped", "was_flipped", "gender_confidence"]])


,text,text_clean,text_flipped,was_flipped,gender_confidence
0,"Hola, soy Patri! Encantada de conocerte","Hola, soy {{name}}! Encantada de conocerte","Hola, soy {{name}}! Encantado de conocerte.",True,0.998070
1,"Holaaa buenas, soy Nombre, acabo de llegar y estoy todavía un poco perdida, y tú?","Holaaa buenas, soy {{name}}, acabo de llegar y estoy todavía un poco perdida, y tú?","Holaas buenas, soy {{name}}, acabo de llegar y estoy todavía un poco perdido, y tú?",True,0.996565
2,"Hola, Laura, encantado.","Hola, Laura, encantado.","Hola, Laura, encantada.",True,0.999179
3,Emmm. Hola.,Emmm. Hola.,Emmm. Hola.,False,0.993859
4,Hola Laura yo soy Rocío encantada,Hola Laura yo soy {{name}} encantada,"Hola Laura, yo soy {{name}} encantado.",True,0.999370
5,"Hol, soy Briana un gusto","Hol, soy {{name}} un gusto","Hol, soy {{name}} un gusto",False,0.980689
6,Hola,Hola,Hola,False,0.998551
7,"Buenas, yo soy Miguel, encantado","Buenas, yo soy {{name}}, encantado","Buenas, yo soy {{name}}, encantado",False,0.847328
8,Encantado soy ...,Encantado soy {{name}},Encantado soy {{name}},False,0.876115
9,"Hola, soy Gianfranco un placer","Hola, soy {{name}} un placer","Hola, soy {{name}} un placer",False,0.993907
